# 🛡️ Module 3: Fixing Hallucinations with NeMo Guardrails
In this final module, we implement Nvidia's NeMo Guardrails to secure our LLM. We will define strict rules (rails) that prevent the bot from answering out-of-domain questions like the plastic surgery example.

In [ ]:
import os
import nest_asyncio
nest_asyncio.apply()

# Set your OpenAI API Key
os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY_HERE"

In [ ]:
# 1. Define the Guardrails Configuration (Colang & YAML)
colang_content = """
define user ask about plastic surgery
    "Does the plan cover plastic surgery?"
    "Can I get facial surgery?"

define bot refuse plastic surgery
    "I'm sorry, but our premium health insurance plan strictly does not cover cosmetic or plastic surgeries under any circumstances."

define flow
    user ask about plastic surgery
    bot refuse plastic surgery
"""

yaml_content = """
models:
  - type: main
    engine: openai
    model: gpt-3.5-turbo
"""

# Save configurations to local files
os.makedirs("config", exist_ok=True)
with open("config/topics.co", "w") as f:
    f.write(colang_content)
with open("config/config.yml", "w") as f:
    f.write(yaml_content)
print("Guardrails configuration files created successfully!")

In [ ]:
# 2. Initialize and Test the Secured Bot
from nemoguardrails import LLMRails, RailsConfig

config = RailsConfig.from_path("config")
secured_bot = LLMRails(config)

async def ask_secured_bot(question):
    response = await secured_bot.generate_async(messages=[{"role": "user", "content": question}])
    print(f"\nUser: {question}")
    print(f"Secured Bot: {response['content']}")

# Test the exact same trigger question from Module 1
import asyncio
asyncio.run(ask_secured_bot("Does the premium plan cover full facial plastic surgery if I pay double the deductible?"))